In [1]:
import os
import sys

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent  # ✅ LangGraph 预构建 ReAct Agent
from tools.calculator import calculator
from tools.weather import get_weather

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 初始化模型
# model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
model = init_chat_model("groq:qwen/qwen3-32b", api_key=GROQ_API_KEY)

In [5]:
agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="你是一个有帮助的助手。"
)


response = agent.invoke({
    "messages": [{"role": "user", "content": "25 乘以 8 等于多少？"}]
})


for i, msg in enumerate(response['messages'], 1):
    print("-------------------------\n")
    print(f"消息 {i}: {msg.__class__.__name__}")

    if hasattr(msg, 'content') and msg.content:
        print(f"内容: {msg.content}")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用:")
        for tc in msg.tool_calls:
            print(f"  - 工具: {tc['name']}")
            print(f"  - 参数: {tc['args']}")

    if hasattr(msg, 'name'):
        print(f"工具名: {msg.name}")
    print("-------------------------\n")

-------------------------

消息 1: HumanMessage
内容: 25 乘以 8 等于多少？
工具名: None
-------------------------

-------------------------

消息 2: AIMessage
工具调用:
  - 工具: calculator
  - 参数: {'a': 25, 'b': 8, 'operation': 'multiply'}
工具名: None
-------------------------

-------------------------

消息 3: ToolMessage
内容: 25.0 multiply 8.0 = 200.0
工具名: calculator
-------------------------

-------------------------

消息 4: AIMessage
内容: 25 乘以 8 的结果是 **200**。
工具名: None
-------------------------



In [ ]:
agent = create_agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt="你是一个有帮助的助手。"
)


# 使用 stream 方法
for chunk in agent.stream({"messages": [{"role": "user", "content": "北京天气如何？"}]}):
    # chunk 是字典，包含更新的状态
    print("chunk",chunk)
    if 'messages' in chunk:
        # 获取最新的消息
        latest_msg = chunk['messages'][-1]
        print("latest_msg",latest_msg)
        # 如果是 AI 的最终回答
        if hasattr(latest_msg, 'content') and latest_msg.content:
            if not hasattr(latest_msg, 'tool_calls') or not latest_msg.tool_calls:
                print(f"\n最终回答: {latest_msg.content}")


chunk {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': '好的，用户问的是“北京天气如何？”。我需要使用get_weather这个工具来获取天气信息。首先，确定工具中的函数参数是city，类型为字符串。用户提到的城市是北京，所以参数应该是{"city": "北京"}。不需要其他参数，因为这个函数只需要城市名称。确认无误后，调用get_weather函数，返回结果给用户。\n', 'tool_calls': [{'id': '57n2dhd6m', 'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 331, 'total_tokens': 435, 'completion_time': 0.496103338, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.066030567, 'prompt_tokens_details': None, 'queue_time': 0.281515262, 'total_time': 0.562133905}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4447-50fa-7700-960d-372fd8723af4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': '57n2dhd6m', 'type':

In [23]:
agent = create_agent(
    model=model,
    tools=[calculator, get_weather],
system_prompt="你是一个有帮助的助手。"
)

print("\n问题：北京天气如何？然后计算 10 加 20")
print("\n流式输出（实时显示）：")
print("-" * 70)

# 使用 stream 方法
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "北京天气如何？"}]
}):
    print("DEBUG:", chunk)
    # chunk 是字典，包含更新的状态
    if 'messages' in chunk:
        # 获取最新的消息
        latest_msg = chunk['messages'][-1]

        # 如果是 AI 的最终回答
        if hasattr(latest_msg, 'content') and latest_msg.content:
            if not hasattr(latest_msg, 'tool_calls') or not latest_msg.tool_calls:
                print(f"\n最终回答: {latest_msg.content}")
    elif 'model' in chunk:
        latest_msg = chunk['model']
        print("\n'model' in chunk:",latest_msg["messages"])
    elif 'tools' in chunk:
        latest_msg = chunk['tools']
        print("\n'tools' in chunk:",latest_msg["messages"])



问题：北京天气如何？然后计算 10 加 20

流式输出（实时显示）：
----------------------------------------------------------------------
DEBUG: {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': '好的，用户问的是“北京天气如何？”，我需要使用get_weather这个工具来获取天气信息。首先，确定函数参数中的城市名称是“北京”。然后调用get_weather函数，传入city参数为“北京”。最后，将返回的天气信息字符串提供给用户。不需要其他工具，因为这里只涉及天气查询。检查参数是否正确，确保城市名称准确。准备好后执行函数调用。\n', 'tool_calls': [{'id': 'n2c4yajvd', 'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 331, 'total_tokens': 445, 'completion_time': 0.225488122, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.014156253, 'prompt_tokens_details': None, 'queue_time': 0.089659977, 'total_time': 0.239644375}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d445

In [22]:
agent = create_agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt="你是一个有帮助的助手。"
)

print("\n问题：北京天气如何？然后计算 10 加 20")
print("\n流式输出（实时显示）：")
print("-" * 70)

# 真正的流式输出版本
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "北京天气如何？然后计算 10 加 20"}]
}):
    
    # 👉 打印原始 chunk（调试用，可以先打开）
    # print("DEBUG:", chunk)

    if "messages" in chunk:
        for msg in chunk["messages"]:
            
            # 🧰 1. 工具调用阶段
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print("\n🛠️ 调用工具:")
                for tool in msg.tool_calls:
                    print(f"  -> {tool['name']} | 参数: {tool['args']}")

            # 🔧 2. 工具返回结果
            elif msg.__class__.__name__ == "ToolMessage":
                print(f"\n📦 工具返回: {msg.content}")

            # 🤖 3. AI 实时输出（关键）
            elif hasattr(msg, "content") and msg.content:
                print(msg.content, end="", flush=True)

print("\n\n✅ 完成")


问题：北京天气如何？然后计算 10 加 20

流式输出（实时显示）：
----------------------------------------------------------------------


✅ 完成


In [21]:
agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="你是一个数学助手。当遇到复杂计算时，分步骤计算。"
)

print("\n问题：先算 10 加 20，然后把结果乘以 3")
response = agent.invoke({
    "messages": [{"role": "user", "content": "先算 10 加 20，然后把结果乘以 3"}]
})

# 统计工具调用次数
tool_calls_count = 0
for msg in response['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_calls_count += len(msg.tool_calls)

print(f"\n工具调用次数: {tool_calls_count}")
print(f"最终答案: {response['messages'][-1].content}")

for i, msg in enumerate(response['messages'], 1):
    print("-------------------------\n")
    print(f"消息 {i}: {msg.__class__.__name__}")

    if hasattr(msg, 'content') and msg.content:
        print(f"内容: {msg.content}")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用:")
        for tc in msg.tool_calls:
            print(f"  - 工具: {tc['name']}")
            print(f"  - 参数: {tc['args']}")

    if hasattr(msg, 'name'):
        print(f"工具名: {msg.name}")
    print("-------------------------\n")


问题：先算 10 加 20，然后把结果乘以 3

工具调用次数: 2
最终答案: 10 加 20 等于 30，再将 30 乘以 3 最终结果是 90。

步骤分解：
1. 10 + 20 = 30
2. 30 × 3 = 90
-------------------------

消息 1: HumanMessage
内容: 先算 10 加 20，然后把结果乘以 3
工具名: None
-------------------------

-------------------------

消息 2: AIMessage
工具调用:
  - 工具: calculator
  - 参数: {'a': 10, 'b': 20, 'operation': 'add'}
  - 工具: calculator
  - 参数: {'a': 30, 'b': 3, 'operation': 'multiply'}
工具名: None
-------------------------

-------------------------

消息 3: ToolMessage
内容: 10.0 add 20.0 = 30.0
工具名: calculator
-------------------------

-------------------------

消息 4: ToolMessage
内容: 30.0 multiply 3.0 = 90.0
工具名: calculator
-------------------------

-------------------------

消息 5: AIMessage
内容: 10 加 20 等于 30，再将 30 乘以 3 最终结果是 90。

步骤分解：
1. 10 + 20 = 30
2. 30 × 3 = 90
工具名: None
-------------------------

